# 04 · Topic modelling (Sentence-BERT + K-means)

Requires `sentence-transformers` and `scikit-learn` (see requirements.txt). 
Mines sentences containing the focus distortion, embeds them, and picks the 
cluster count minimising the Davies-Bouldin index — all in `src/`.


In [ ]:
# Make `src` importable when running the notebook from the notebooks/ folder.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.config import load_config
cfg = load_config()
COMMUNITY = 'sgexams'  # change to 'teenagers' to analyse the other community

In [ ]:
from src.cognitive_distortions.target_words import DISTORTIONS
from src.topic_modeling.embed_cluster import run_topic_model
from src.data.datasets import load_community
focus = cfg.get('topic_modeling.focus_category', 'Emotional Reasoning')
data = load_community(cfg, COMMUNITY)
texts = data['comments']['body'].dropna().astype(str).tolist()
k_range = range(cfg.get('topic_modeling.k_min', 10), cfg.get('topic_modeling.k_max', 50))
result = run_topic_model(texts, DISTORTIONS[focus],
                         model_name=cfg.get('topic_modeling.embedding_model'),
                         k_range=k_range, context=cfg.get('topic_modeling.context', 'sentence'))
print('optimal_k =', result['optimal_k'])
result['examples'].head()

In [ ]:
from src.visualization.plots import davies_bouldin_plot, cluster_frequency_bar
davies_bouldin_plot({focus: result['scores']}, list(k_range))

In [ ]:
cluster_frequency_bar(result['labels'])